In [1]:
# ==========================================================
# ALGORITHME C5.0 FROM SCRATCH EN PYTHON
# ==========================================================
#
# C5.0 = amélioration de C4.5
#
# ----------------------------------------------------------
# C5.0 EST UTILISÉ POUR :
# ----------------------------------------------------------
#
# construire des arbres de décision
# plus rapides et plus performants.
#
# ----------------------------------------------------------
# DIFFÉRENCE ENTRE :
# ----------------------------------------------------------
#
# ID3 :
# -> utilise Information Gain
#
# C4.5 :
# -> utilise Gain Ratio
#
# C5.0 :
# -> amélioration de C4.5
# -> plus rapide
# -> moins de mémoire
# -> pruning amélioré
# -> supporte boosting
#
# ----------------------------------------------------------
# DANS CE CODE ON IMPLÉMENTE :
# ----------------------------------------------------------
#
# ✔ Entropie
# ✔ Information Gain
# ✔ Split Info
# ✔ Gain Ratio
# ✔ Construction arbre
# ✔ Prédiction
# ✔ Pruning simple
#
# ==========================================================

# ==========================================================
# IMPORTATION DES LIBRARIES
# ==========================================================

# pandas :
# manipulation des tableaux de données
import pandas as pd

# numpy :
# calculs mathématiques
import numpy as np

# Counter :
# compter les occurrences
from collections import Counter

# ==========================================================
# 1. DATASET
# ==========================================================


# Dataset :
#
# Objectif :
# prédire si on joue au tennis
#
# Variable cible :
# Jouer = Oui / Non
#
# ----------------------------------------------------------

data = {

    # ------------------------------------------------------
    # FEATURE : MÉTÉO
    # ------------------------------------------------------
    #
    # Valeurs possibles :
    #
    # Soleil
    # Nuageux
    # Pluie
    
    # ------------------------------------------------------

    'Meteo': [

        'Soleil',
        'Soleil',
        'Nuageux',
        'Pluie',
        'Pluie',
        'Pluie',
        'Nuageux',
        'Soleil',
        'Soleil',
        'Pluie'
    ],

    # ------------------------------------------------------
    # FEATURE : TEMPÉRATURE
    # ------------------------------------------------------

    'Temperature': [

        'Chaud',
        'Chaud',
        'Chaud',
        'Moyen',
        'Froid',
        'Froid',
        'Froid',
        'Moyen',
        'Froid',
        'Moyen'
    ],

    # ------------------------------------------------------
    # FEATURE : HUMIDITÉ
    # ------------------------------------------------------

    'Humidite': [

        'Haute',
        'Haute',
        'Haute',
        'Haute',
        'Normale',
        'Normale',
        'Normale',
        'Haute',
        'Normale',
        'Normale'
    ],

    # ------------------------------------------------------
    # FEATURE : VENT
    # ------------------------------------------------------

    'Vent': [

        'Faible',
        'Fort',
        'Faible',
        'Faible',
        'Faible',
        'Fort',
        'Fort',
        'Faible',
        'Faible',
        'Faible'
    ],

    # ------------------------------------------------------
    # VARIABLE CIBLE
    # ------------------------------------------------------
    #
    # Oui = jouer
    # Non = ne pas jouer
    #
    # ------------------------------------------------------

    'Jouer': [

        'Non',
        'Non',
        'Oui',
        'Oui',
        'Oui',
        'Non',
        'Oui',
        'Non',
        'Oui',
        'Oui'
    ]
}

# ----------------------------------------------------------
# Transformer dictionnaire -> DataFrame
# ----------------------------------------------------------

df = pd.DataFrame(data)

# ----------------------------------------------------------
# Afficher dataset
# ----------------------------------------------------------

print("===== DATASET =====")

print(df)

# ==========================================================
# 2. ENTROPIE
# ==========================================================

def entropy(y):

    """
    ------------------------------------------------------
    ENTROPIE
    ------------------------------------------------------

    L'entropie mesure le désordre
    dans les données.

    ------------------------------------------------------
    CAS 1 :
    ------------------------------------------------------

    Oui Oui Oui Oui

    -> faible entropie
    -> données pures

    ------------------------------------------------------
    CAS 2 :
    ------------------------------------------------------

    Oui Non Oui Non

    -> forte entropie
    -> données mélangées

    ------------------------------------------------------
    FORMULE :
    ------------------------------------------------------

    Entropy(S) = - Σ p(x) log2(p(x))

    ------------------------------------------------------
    p(x)
    ------------------------------------------------------

    probabilité d'une classe

    Exemple :

    Oui = 6/10
    Non = 4/10
    """

    # ------------------------------------------------------
    # Compter occurrences des classes
    # ------------------------------------------------------
    #
    # Exemple :
    #
    # Oui = 6
    # Non = 4
    #
    # ------------------------------------------------------

    counts = Counter(y)

    # ------------------------------------------------------
    # Nombre total d'exemples
    # ------------------------------------------------------

    total = len(y)

    # ------------------------------------------------------
    # Variable entropie
    # ------------------------------------------------------

    ent = 0

    # ------------------------------------------------------
    # Parcourir chaque classe
    # ------------------------------------------------------

    for count in counts.values():

        # --------------------------------------------------
        # Calcul probabilité
        # --------------------------------------------------

        p = count / total

        # --------------------------------------------------
        # Formule entropie
        # --------------------------------------------------

        ent -= p * np.log2(p)

    # ------------------------------------------------------
    # Retourner entropie finale
    # ------------------------------------------------------

    return ent

# ==========================================================
# 3. INFORMATION GAIN
# ==========================================================

def information_gain(data, feature, target):

    """
    ------------------------------------------------------
    INFORMATION GAIN
    ------------------------------------------------------

    Mesure combien une feature
    réduit le désordre.

    ------------------------------------------------------
    PLUS LE GAIN EST GRAND :
    ------------------------------------------------------

    -> meilleure séparation

    -> meilleure feature

    ------------------------------------------------------
    FORMULE :
    ------------------------------------------------------

    Gain =
    Entropie totale
    -
    Entropie pondérée
    """

    # ------------------------------------------------------
    # ÉTAPE 1 :
    # calcul entropie totale
    # ------------------------------------------------------

    total_entropy = entropy(data[target])

    # ------------------------------------------------------
    # Récupérer valeurs uniques
    # ------------------------------------------------------

    values = data[feature].unique()

    # ------------------------------------------------------
    # Variable entropie pondérée
    # ------------------------------------------------------

    weighted_entropy = 0

    # ------------------------------------------------------
    # Parcourir chaque valeur
    # ------------------------------------------------------

    for value in values:

        # --------------------------------------------------
        # Créer sous-ensemble
        # --------------------------------------------------
        #
        # Exemple :
        #
        # Meteo = Soleil
        #
        # --------------------------------------------------

        subset = data[data[feature] == value]

        # --------------------------------------------------
        # Entropie sous-ensemble
        # --------------------------------------------------

        subset_entropy = entropy(subset[target])

        # --------------------------------------------------
        # Calcul poids
        # --------------------------------------------------
        #
        # poids =
        #
        # taille sous-ensemble
        # -------------------
        # taille dataset
        #
        # --------------------------------------------------

        weight = len(subset) / len(data)

        # --------------------------------------------------
        # Ajouter entropie pondérée
        # --------------------------------------------------

        weighted_entropy += weight * subset_entropy

    # ------------------------------------------------------
    # Calcul gain final
    # ------------------------------------------------------

    gain = total_entropy - weighted_entropy

    return gain

# ==========================================================
# 4. SPLIT INFO
# ==========================================================

def split_info(data, feature):

    """
    ------------------------------------------------------
    SPLIT INFO
    ------------------------------------------------------

    Split Info mesure :
    la dispersion des divisions.

    ------------------------------------------------------
    POURQUOI ?
    ------------------------------------------------------

    Certaines features créent
    trop de branches.

    Exemple :

    ID étudiant :
    1
    2
    3
    4
    ...

    -> une branche par étudiant

    -> mauvais comportement

    ------------------------------------------------------
    C5.0 pénalise ces features
    grâce au Split Info.
    ------------------------------------------------------
    """

    # ------------------------------------------------------
    # Taille dataset
    # ------------------------------------------------------

    total = len(data)

    # ------------------------------------------------------
    # Valeurs uniques
    # ------------------------------------------------------

    values = data[feature].unique()

    # ------------------------------------------------------
    # Variable SplitInfo
    # ------------------------------------------------------

    split = 0

    # ------------------------------------------------------
    # Parcourir chaque valeur
    # ------------------------------------------------------

    for value in values:

        # --------------------------------------------------
        # Créer sous-ensemble
        # --------------------------------------------------

        subset = data[data[feature] == value]

        # --------------------------------------------------
        # Calcul probabilité
        # --------------------------------------------------

        p = len(subset) / total

        # --------------------------------------------------
        # Formule SplitInfo
        # --------------------------------------------------

        split -= p * np.log2(p)

    return split

# ==========================================================
# 5. GAIN RATIO
# ==========================================================

def gain_ratio(data, feature, target):

    """
    ------------------------------------------------------
    GAIN RATIO
    ------------------------------------------------------

    Gain Ratio améliore
    Information Gain.

    ------------------------------------------------------
    FORMULE :
    ------------------------------------------------------

    GainRatio =
    Gain
    -----
    SplitInfo

    ------------------------------------------------------
    OBJECTIF :
    ------------------------------------------------------

    éviter de favoriser
    les features avec beaucoup
    de valeurs différentes.
    """

    # ------------------------------------------------------
    # Calcul Information Gain
    # ------------------------------------------------------

    gain = information_gain(
        data,
        feature,
        target
    )

    # ------------------------------------------------------
    # Calcul Split Info
    # ------------------------------------------------------

    split = split_info(
        data,
        feature
    )

    # ------------------------------------------------------
    # Éviter division par zéro
    # ------------------------------------------------------

    if split == 0:

        return 0

    # ------------------------------------------------------
    # Retourner Gain Ratio
    # ------------------------------------------------------

    return gain / split

# ==========================================================
# 6. CHOISIR MEILLEURE FEATURE
# ==========================================================

def best_feature(data, features, target):

    """
    ------------------------------------------------------
    Cette fonction teste
    toutes les features
    puis choisit celle qui possède
    le plus grand Gain Ratio.
    ------------------------------------------------------
    """

    # ------------------------------------------------------
    # Dictionnaire des ratios
    # ------------------------------------------------------

    ratios = {}

    print("\n===== GAIN RATIO =====")

    # ------------------------------------------------------
    # Calcul ratio pour chaque feature
    # ------------------------------------------------------

    for feature in features:

        ratio = gain_ratio(
            data,
            feature,
            target
        )

        ratios[feature] = ratio

        print(f"{feature} : {ratio:.4f}")

    # ------------------------------------------------------
    # Retourner meilleure feature
    # ------------------------------------------------------

    return max(ratios, key=ratios.get)

# ==========================================================
# 7. PRUNING SIMPLE
# ==========================================================

def should_prune(data, target, threshold=0.90):

    """
    ------------------------------------------------------
    PRUNING
    ------------------------------------------------------

    Le pruning sert à :
    simplifier l'arbre.

    ------------------------------------------------------
    POURQUOI ?
    ------------------------------------------------------

    Un arbre trop profond peut :

    -> mémoriser les données
    -> faire du surapprentissage
    -> mal généraliser

    ------------------------------------------------------
    IDÉE :
    ------------------------------------------------------

    Si une classe domine fortement,
    on arrête les divisions.

    Exemple :

    Oui = 95%
    Non = 5%

    -> inutile de continuer

    ------------------------------------------------------
    threshold = 0.90
    ------------------------------------------------------

    Si une classe représente
    au moins 90%
    alors :
    -> on coupe l'arbre
    """

    # ------------------------------------------------------
    # Compter classes
    # ------------------------------------------------------

    counts = Counter(data[target])

    # ------------------------------------------------------
    # Nombre total d'exemples
    # ------------------------------------------------------

    total = len(data)

    # ------------------------------------------------------
    # Classe majoritaire
    # ------------------------------------------------------

    majority = max(counts.values()) / total

    # ------------------------------------------------------
    # Retourner True ou False
    # ------------------------------------------------------

    return majority >= threshold

# ==========================================================
# 8. CONSTRUCTION ARBRE C5.0
# ==========================================================

def c50(data, features, target):

    """
    ------------------------------------------------------
    Fonction récursive principale
    ------------------------------------------------------

    Elle construit l'arbre
    étape par étape.
    """

    # ------------------------------------------------------
    # Classes présentes
    # ------------------------------------------------------

    labels = data[target]

    # ======================================================
    # CAS D'ARRÊT 1
    # ======================================================
    #
    # Toutes les classes identiques
    #
    # Exemple :
    #
    # Oui Oui Oui
    #
    # -> feuille finale
    #
    # ======================================================

    if len(np.unique(labels)) == 1:

        return labels.iloc[0]

    # ======================================================
    # CAS D'ARRÊT 2
    # ======================================================
    #
    # PRUNING
    #
    # Si une classe domine fortement,
    # on arrête l'arbre.
    #
    # ======================================================

    if should_prune(data, target):

        return labels.mode()[0]

    # ======================================================
    # CAS D'ARRÊT 3
    # ======================================================
    #
    # Plus de features disponibles
    #
    # ======================================================

    if len(features) == 0:

        return labels.mode()[0]

    # ======================================================
    # ÉTAPE 1 :
    # choisir meilleure feature
    # ======================================================

    best = best_feature(
        data,
        features,
        target
    )

    # ======================================================
    # Création arbre
    # ======================================================

    tree = {best: {}}

    # ======================================================
    # Valeurs possibles de la feature
    # ======================================================

    values = data[best].unique()

    # ======================================================
    # Construction des branches
    # ======================================================

    for value in values:

        # --------------------------------------------------
        # Sous-ensemble correspondant
        # --------------------------------------------------

        subset = data[data[best] == value]

        # --------------------------------------------------
        # Supprimer feature utilisée
        # --------------------------------------------------

        remaining_features = [

            f for f in features

            if f != best
        ]

        # --------------------------------------------------
        # APPEL RÉCURSIF
        # --------------------------------------------------
        #
        # construire sous-arbre
        #
        # --------------------------------------------------

        subtree = c50(
            subset,
            remaining_features,
            target
        )

        # --------------------------------------------------
        # Ajouter sous-arbre
        # --------------------------------------------------

        tree[best][value] = subtree

    # ------------------------------------------------------
    # Retourner arbre final
    # ------------------------------------------------------

    return tree

# ==========================================================
# 9. ENTRAINEMENT
# ==========================================================

# ----------------------------------------------------------
# Liste des features
# ----------------------------------------------------------

features = [

    'Meteo',
    'Temperature',
    'Humidite',
    'Vent'
]

# ----------------------------------------------------------
# Variable cible
# ----------------------------------------------------------

target = 'Jouer'

# ----------------------------------------------------------
# Construction arbre
# ----------------------------------------------------------

tree = c50(
    df,
    features,
    target
)

# ==========================================================
# 10. AFFICHAGE
# ==========================================================

print("\n===== ARBRE C5.0 =====")

print(tree)

# ==========================================================
# 11. PRÉDICTION
# ==========================================================

def predict(tree, sample):

    """
    ------------------------------------------------------
    Fonction de prédiction
    ------------------------------------------------------

    Elle parcourt l'arbre
    jusqu'à trouver
    une feuille finale.
    """

    # ------------------------------------------------------
    # Récupérer racine
    # ------------------------------------------------------

    root = list(tree.keys())[0]

    # ------------------------------------------------------
    # Valeur correspondante
    # dans l'exemple
    # ------------------------------------------------------

    value = sample[root]

    # ------------------------------------------------------
    # Aller dans branche correspondante
    # ------------------------------------------------------

    subtree = tree[root][value]

    # ------------------------------------------------------
    # Si feuille finale
    # ------------------------------------------------------

    if not isinstance(subtree, dict):

        return subtree

    # ------------------------------------------------------
    # Sinon continuer récursivement
    # ------------------------------------------------------

    return predict(subtree, sample)

# ==========================================================
# 12. TEST
# ==========================================================

# ----------------------------------------------------------
# Nouvel exemple
# ----------------------------------------------------------

sample = {

    'Meteo': 'Pluie',

    'Temperature': 'Froid',

    'Humidite': 'Normale',

    'Vent': 'Faible'
}

# ----------------------------------------------------------
# Faire prédiction
# ----------------------------------------------------------

prediction = predict(
    tree,
    sample
)

# ==========================================================
# 13. AFFICHAGE RÉSULTAT
# ==========================================================

print("\n===== TEST =====")

print("Exemple :", sample)

print("Classe prédite :", prediction)

===== DATASET =====
     Meteo Temperature Humidite    Vent Jouer
0   Soleil       Chaud    Haute  Faible   Non
1   Soleil       Chaud    Haute    Fort   Non
2  Nuageux       Chaud    Haute  Faible   Oui
3    Pluie       Moyen    Haute  Faible   Oui
4    Pluie       Froid  Normale  Faible   Oui
5    Pluie       Froid  Normale    Fort   Non
6  Nuageux       Froid  Normale    Fort   Oui
7   Soleil       Moyen    Haute  Faible   Non
8   Soleil       Froid  Normale  Faible   Oui
9    Pluie       Moyen  Normale  Faible   Oui

===== GAIN RATIO =====
Meteo : 0.2115
Temperature : 0.0608
Humidite : 0.1245
Vent : 0.1036

===== GAIN RATIO =====
Temperature : 0.5409
Humidite : 1.0000
Vent : 0.1511

===== GAIN RATIO =====
Temperature : 0.3113
Humidite : 0.1511
Vent : 1.0000

===== ARBRE C5.0 =====
{'Meteo': {'Soleil': {'Humidite': {'Haute': 'Non', 'Normale': 'Oui'}}, 'Nuageux': 'Oui', 'Pluie': {'Vent': {'Faible': 'Oui', 'Fort': 'Non'}}}}

===== TEST =====
Exemple : {'Meteo': 'Pluie', 'Temperature':